In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from loss import PDE_loss, dirichlet_loss, all_at_once_loss
from utils import get_knots
from model import PINN_Net, PINN_Net_Fourier, FourierFeatureLayer, ScaledTanh, PINN_Net_Adapt, P2INN
from data_prep import sample_collocation, sample_terminal, PINNDataFactory, PINNAdaptiveDataFactory
from torch.utils.data import DataLoader
from utils import batch_get_difference, grad_net, Simpson_rule, trapz_rule, grad_log_w, simulate_samples, RingCoupledDoubleWell, diff_function
import argparse
import os
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve
from itertools import count
from sklearn.linear_model import LinearRegression, Ridge

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

In [ ]:
T = 5
# K = 2
n = 500
dt = T / n
t_0 = 0.0
t_grid = get_knots(T, n)
kappa = [1.3839, 0.9836, 1.1328, 1.1143]
sigma = 0.4
# mu = 0
# sigma = 0.5
r = 0.2
x_min = 0.0
x_max = 2.5
batch_size = 1000
n_points = 101
x0 = torch.tensor([1.5, 1.5, 1.5, 1.5], device="cpu")
d = x0.shape[-1]
drift_func = RingCoupledDoubleWell(kappa, 0.5)
diff_func = diff_function(sigma)
Sigma = diff_func(x0)

print(Sigma)

In [ ]:
_, X, Y = simulate_samples(RingCoupledDoubleWell([1.0, 1.0, 1.0, 1.0], 0.5), diff_func, T, n, -10.0, 10.0, X0=x0, seed=7)
obs_times = [0.0, 1.0, 2.0, 3.0, 4.0, 5.0]
obs_idx = [t / dt for t in obs_times]
x = X[:, obs_idx]
y = Y[:, obs_idx]
N = len(obs_times) - 1
print(y)
print(N)
print(X)

In [ ]:
models = nn.ModuleList([PINN_Net(d, 4, 70) for _ in range(N+1)]).to(device)

In [ ]:
data_factory = PINNDataFactory(obs_times, x_min, x_max, d, 1000, 1000, dtype=torch.float32, device="cpu")

In [ ]:
def PINN_train(nets, drift_func, diff_func, data_factory, early_stop_threshold, not_in_EM=True):
    optimizer = torch.optim.AdamW(nets.parameters(), lr=1e-4)
    consecutive_below = 0

    for epoch in count(0):

        optimizer.zero_grad(set_to_none=True)
        interval_loaders, knot_loaders = data_factory()
        for net in nets:
            net.train()

        loss = all_at_once_loss(interval_loaders, knot_loaders, nets, drift_func, diff_func, y=y, r=r, solve_for="logw")
        loss.backward()
        optimizer.step()

        loss_val = float(loss.detach().cpu())
        if not_in_EM:
            print(f"epoch {epoch}: loss = {loss_val:.4f}")

        if loss_val < early_stop_threshold:
            consecutive_below += 1
        else:
            consecutive_below = 0   

        if consecutive_below > 3:
            break   

In [ ]:
ckpt_path = "4DDoubleWell_5_[1.3839, 0.9836, 1.1328, 1.1143]_xmin0.0_max2.5_all_at_once_diff0.4_initial_seed7/pinn_models_all_at_once.pt"
state = torch.load(ckpt_path, map_location="cpu")
models.load_state_dict(state["models_state_dict"])
models.to(device)

In [ ]:
PINN_train(models, drift_func, diff_func, data_factory, early_stop_threshold=20.0)

In [ ]:
save_dir = "4DDoubleWell_5_[1.3839, 0.9836, 1.1328, 1.1143]_xmin0.0_max2.5_all_at_once_diff0.4_initial_seed7"
os.makedirs(save_dir, exist_ok=True)
ckpt_path = os.path.join(save_dir, f"pinn_models_all_at_once.pt")

ckpt = {
    "models_state_dict": models.state_dict(),  # works for ModuleList (recursive)
    # (optional) include anything else you want to remember:
    "meta": {"N": N, "in_dim": 1, "out_dim": 5, "hidden": 70}
}
torch.save(ckpt, ckpt_path)
print(f"Saved: {ckpt_path}")

In [ ]:
def mle_kappa(X, dt, k):
    """
    Computes the exact closed-form MLE (M-step update) for the kappa parameters 
    in the 4D Ring-Coupled Double-Well model.
    
    Args:
        X: torch.Tensor of shape (n_paths, M_steps, 4) representing the simulated trajectories.
        dt: float, time step size used in the simulation (e.g., 0.01).
        k: float, the fixed coupling strength constant.
        
    Returns:
        kappa_hat: torch.Tensor of shape (4,) containing the updated estimates for kappa.
    """
    # 1. Left-point evaluation for Ito integral (all steps except the very last)
    X_t = X[:, :-1, :]  # Shape: (n_paths, M_steps - 1, 4)
    
    # 2. Forward increments (dX)
    dX = X[:, 1:, :] - X[:, :-1, :]  # Shape: (n_paths, M_steps - 1, 4)
    
    # 3. Calculate the non-kappa drift components: f(X) = -X^3 + k * (X_left + X_right - 2X)
    # torch.roll efficiently handles the periodic boundary conditions of the ring graph
    X_left = torch.roll(X_t, shifts=1, dims=-1)
    X_right = torch.roll(X_t, shifts=-1, dims=-1)
    
    f_X = -torch.pow(X_t, 3) + k * (X_left + X_right - 2 * X_t)
    
    # 4. Numerator: Sum of (dX - f(X)*dt) * X_t across paths (dim=0) and time steps (dim=1)
    # This computes the covariance between the unexplained increment and the state.
    numerator = torch.sum((dX - f_X * dt) * X_t, dim=(0, 1))
    
    # 5. Denominator: Sum of (X_t^2) * dt across paths and time steps
    # This serves as the normalization factor (integrated variance).
    denominator = torch.sum((X_t ** 2) * dt, dim=(0, 1))
    
    # 6. Element-wise division to yield the 4 kappa values
    kappa_hat = numerator / denominator
    
    return kappa_hat.tolist()

In [ ]:
posterior_drift_func = lambda x, t: drift_func(x, t) + diff_func(x)**2 * grad_log_w(x, t, models, obs_times)
t_grid, X, _ = simulate_samples(posterior_drift_func, diff_func, T, 100, x_min, x_max, X0=x0, n_paths=1, clamp=False)
print(X)
x1 = torch.tensor([1.4553, 1.5728, 1.4333, 1.4040], device="cpu")

In [ ]:
def EM_algorithm(iteration, models, kappa, T, n, X0, num_path):
    kappa1_list = [kappa[0]]
    kappa2_list = [kappa[1]]
    kappa3_list = [kappa[2]]
    kappa4_list = [kappa[3]]
    # kappa5_list = [kappa[4]]
    prior_drift_func = RingCoupledDoubleWell(kappa, 0.5)

    for iter in range(iteration):
        PINN_train(models, prior_drift_func, diff_func, data_factory=data_factory, early_stop_threshold=30.00, not_in_EM=False)
        # models = [net.to("cpu") for net in models]
        posterior_drift_func = lambda x, t: prior_drift_func(x, t) + 0.05 * diff_func(x)**2 * grad_log_w(x, t, models, obs_times)
        t_grid, X, _ = simulate_samples(posterior_drift_func, diff_func, T, n, x_min, x_max, X0=X0, n_paths=num_path, clamp=True)
        dt = t_grid[1] - t_grid[0]
        dt, X = dt.to("cpu"), X.to("cpu")
        kappa_new = mle_kappa(X, dt, 0.5)
        # kappa1 = (1 - alpha) * kappa1_hat + alpha * kappa1
        # kappa_minus1 = (1 - alpha) * kappa_minus1_hat + alpha * kappa_minus1
        prior_drift_func.update(kappa=kappa_new)

        kappa1_list.append(kappa_new[0])
        kappa2_list.append(kappa_new[1])
        kappa3_list.append(kappa_new[2])
        kappa4_list.append(kappa_new[3])
        print(f"iter {iter}: k1 is {kappa_new[0]:.4f}, k2 is {kappa_new[1]:.4f}, k3 is {kappa_new[2]:.4f}, k4 is {kappa_new[3]:.4f}")
    
    return kappa1_list, kappa2_list, kappa3_list, kappa4_list

In [ ]:
_, _, _, _ = EM_algorithm(1000, models, kappa, T, n, x0, 100)